# Module 06 — Lecture 1: cuBLAS for Neural Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_06_advanced_topics/01_cublas_neural_models.ipynb)

---

Hand-written kernels give full control but rarely reach peak FLOP efficiency. For **dense matrix operations** — weight matrix multiplies in rate networks, covariance computation, PCA of spike trains — **cuBLAS** delivers vendor-optimized routines that exploit tensor cores and achieve 90%+ of theoretical peak performance.

**Learning objectives:**
- Understand the cuBLAS API and column-major convention
- Use `cublasSgemm` for the forward pass of a rate-coded neural network
- Compute spike-train covariance matrices on GPU
- Compare cuBLAS vs hand-written GEMM performance

In [ ]:
!nvidia-smi

## 1. cuBLAS Fundamentals

### The Column-Major Gotcha

cuBLAS assumes **column-major** (Fortran) storage — columns are contiguous. C/Python use **row-major** (C order). For a matrix A with shape (M×N):

```
Row-major (C):    A[i][j] = data[i*N + j]   ← rows are contiguous
Col-major (BLAS): A[i][j] = data[j*M + i]   ← columns are contiguous
```

**Practical solution:** Compute `C = A * B` in row-major by telling cuBLAS to compute `C^T = B^T * A^T` in column-major:

```c
// Row-major A(M×K) * B(K×N) = C(M×N)
// cuBLAS sees: col-major B^T(N×K) * A^T(K×M) = C^T(N×M)
cublasSgemm(handle,
    CUBLAS_OP_N, CUBLAS_OP_N,
    N, M, K,      // (cols_C, rows_C, shared)
    &alpha,
    d_B, N,       // leading dimension of B^T = N
    d_A, K,       // leading dimension of A^T = K
    &beta,
    d_C, N);      // leading dimension of C^T = N
```

### Key Functions

| Function | Operation | Use case |
|----------|-----------|----------|
| `cublasSgemm` | C = α·A·B + β·C | Weight multiply |
| `cublasSgemv` | y = α·A·x + β·y | Single-timestep forward pass |
| `cublasSger`  | A = α·x·y^T + A | Outer product (Hebbian update) |
| `cublasSdot`  | scalar = x·y | Dot product |
| `cublasSnrm2` | scalar = ‖x‖ | L2 norm |

In [ ]:
%%writefile cublas_neuro.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)

#define CUBLAS_CHECK(call) do { cublasStatus_t e=(call); \
    if(e!=CUBLAS_STATUS_SUCCESS){fprintf(stderr,"cuBLAS error %d\n",e);exit(1);}} while(0)

// Custom GEMM kernel from Module 02 (for comparison)
#define TILE 16
__global__ void matmul_tiled(const float* A, const float* B, float* C,
                              int M, int N, int K)
{
    __shared__ float As[TILE][TILE], Bs[TILE][TILE];
    int row = blockIdx.y*TILE + threadIdx.y;
    int col = blockIdx.x*TILE + threadIdx.x;
    float acc = 0.f;
    for (int t = 0; t < (K + TILE - 1) / TILE; t++) {
        As[threadIdx.y][threadIdx.x] =
            (row < M && t*TILE+threadIdx.x < K) ? A[row*K + t*TILE+threadIdx.x] : 0.f;
        Bs[threadIdx.y][threadIdx.x] =
            (col < N && t*TILE+threadIdx.y < K) ? B[(t*TILE+threadIdx.y)*N + col] : 0.f;
        __syncthreads();
        for (int k = 0; k < TILE; k++) acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }
    if (row < M && col < N) C[row*N + col] = acc;
}

int main(int argc, char** argv)
{
    // Rate network forward pass: out[M × T] = W[M × N] * rates[N × T]
    // M = output neurons, N = input neurons, T = time steps (batch)
    int M = (argc>1) ? atoi(argv[1]) : 1024;
    int N = (argc>2) ? atoi(argv[2]) : 1024;
    int T = (argc>3) ? atoi(argv[3]) : 256;

    printf("Rate network forward pass: W[%d×%d] * r[%d×%d]\n", M, N, N, T);

    float *h_W=(float*)malloc((size_t)M*N*sizeof(float));
    float *h_r=(float*)malloc((size_t)N*T*sizeof(float));
    srand(42);
    for(int i=0;i<M*N;i++) h_W[i]=(float)rand()/RAND_MAX*0.1f;
    for(int i=0;i<N*T;i++) h_r[i]=(float)rand()/RAND_MAX;

    float *d_W,*d_r,*d_out_cublas,*d_out_custom;
    CUDA_CHECK(cudaMalloc(&d_W,         (size_t)M*N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_r,         (size_t)N*T*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_out_cublas,(size_t)M*T*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_out_custom,(size_t)M*T*sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_W,h_W,(size_t)M*N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_r,h_r,(size_t)N*T*sizeof(float),cudaMemcpyHostToDevice));

    cublasHandle_t handle;
    CUBLAS_CHECK(cublasCreate(&handle));

    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    int REPS=100;
    float alpha=1.f, beta=0.f;

    // cuBLAS: C(M×T) = W(M×N) * r(N×T)
    // Col-major trick: compute C^T(T×M) = r^T(T×N) * W^T(N×M)
    CUDA_CHECK(cudaEventRecord(t0));
    for(int rep=0;rep<REPS;rep++) {
        CUBLAS_CHECK(cublasSgemm(handle,
            CUBLAS_OP_N, CUBLAS_OP_N,
            T, M, N,
            &alpha, d_r, T, d_W, N, &beta, d_out_cublas, T));
    }
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    float cublas_ms = ms/REPS;
    float flops = 2.f*M*N*T;
    printf("cuBLAS:  %.3f ms, %.1f GFLOPS\n", cublas_ms, flops/(cublas_ms*1e6f));

    // Custom tiled kernel
    dim3 block(TILE,TILE);
    dim3 grid((T+TILE-1)/TILE, (M+TILE-1)/TILE);
    CUDA_CHECK(cudaEventRecord(t0));
    for(int rep=0;rep<REPS;rep++) {
        matmul_tiled<<<grid,block>>>(d_W, d_r, d_out_custom, M, T, N);
    }
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    float custom_ms = ms/REPS;
    printf("Custom:  %.3f ms, %.1f GFLOPS (%.1fx slower than cuBLAS)\n",
           custom_ms, flops/(custom_ms*1e6f), custom_ms/cublas_ms);

    CUBLAS_CHECK(cublasDestroy(handle));
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_W); cudaFree(d_r); cudaFree(d_out_cublas); cudaFree(d_out_custom);
    free(h_W); free(h_r);
    return 0;
}

In [ ]:
!nvcc -O2 -o cublas_neuro cublas_neuro.cu -lcublas -lm
!echo "--- 1024×1024×256 ---" && ./cublas_neuro 1024 1024 256
!echo "--- 2048×2048×512 ---" && ./cublas_neuro 2048 2048 512

## 2. Performance Sweep

cuBLAS performance improves dramatically with matrix size because:
- Larger tiles → higher arithmetic intensity (more reuse from shared memory)
- Better utilization of tensor cores (available on Volta/Turing/Ampere)
- Amortized kernel launch overhead

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt

sizes = [128, 256, 512, 1024, 2048]
cublas_gflops = []
custom_gflops = []

for N in sizes:
    result = subprocess.run(['./cublas_neuro', str(N), str(N), str(N)],
                            capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if 'cuBLAS' in line:
            cublas_gflops.append(float(line.split()[2]))
        elif 'Custom' in line:
            custom_gflops.append(float(line.split()[2]))

# T4 theoretical peak
T4_fp32_peak = 8100  # GFLOPS

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sizes, cublas_gflops, 'b-o', lw=2, markersize=8, label='cuBLAS')
ax.plot(sizes, custom_gflops, 'r-s', lw=2, markersize=8, label='Custom tiled (TILE=16)')
ax.axhline(T4_fp32_peak, color='gray', linestyle='--', alpha=0.6, label='T4 FP32 peak')
ax.set_xlabel('Matrix size N (N×N × N)', fontsize=12)
ax.set_ylabel('GFLOPS', fontsize=12)
ax.set_title('cuBLAS vs Custom GEMM Performance', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_xticks(sizes); ax.set_xticklabels(sizes)

plt.tight_layout()
plt.savefig('cublas_perf.png', dpi=150, bbox_inches='tight')
plt.show()

for n, c, m in zip(sizes, cublas_gflops, custom_gflops):
    print(f"N={n:4d}: cuBLAS={c:6.0f} GFLOPS, custom={m:6.0f} GFLOPS, "
          f"speedup={c/m:.1f}x, efficiency={c/T4_fp32_peak*100:.0f}%")

## 3. Spike-Train Covariance

Computing the covariance matrix of spike trains is a common neuroscience operation:
$$C_{ij} = \frac{1}{T} \sum_{t=1}^{T} s_i(t) \cdot s_j(t)$$

In matrix form: $\mathbf{C} = \frac{1}{T} \mathbf{S} \mathbf{S}^\top$

This is exactly a GEMM. With N=200 neurons and T=10,000 bins, this is a 200×10,000 × 10,000×200 multiply — 8×10⁹ FLOPs that cuBLAS handles in milliseconds.

In [ ]:
%%writefile spike_cov.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s %d\n",cudaGetErrorString(e),__LINE__);exit(1);}} while(0)
#define CUBLAS_CHECK(call) do { cublasStatus_t e=(call); \
    if(e!=CUBLAS_STATUS_SUCCESS){fprintf(stderr,"cuBLAS %d\n",e);exit(1);}} while(0)

int main(int argc, char** argv) {
    int Nn = (argc>1) ? atoi(argv[1]) : 200;
    int Tb = (argc>2) ? atoi(argv[2]) : 10000;
    float rate = 0.05f;

    // Synthetic spike matrix S[Nn × Tb]
    float* h_S=(float*)calloc((size_t)Nn*Tb,sizeof(float));
    srand(42);
    for(int i=0;i<Nn*Tb;i++) if((float)rand()/RAND_MAX < rate) h_S[i]=1.f;
    int n_spikes=0;
    for(int i=0;i<Nn*Tb;i++) n_spikes+=(int)h_S[i];
    printf("S[%d×%d], spikes=%d (%.1f%%)\n",Nn,Tb,n_spikes,100.f*n_spikes/(Nn*Tb));

    float *d_S,*d_C;
    CUDA_CHECK(cudaMalloc(&d_S,(size_t)Nn*Tb*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_C,(size_t)Nn*Nn*sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_S,h_S,(size_t)Nn*Tb*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_C,0,(size_t)Nn*Nn*sizeof(float)));

    cublasHandle_t handle; CUBLAS_CHECK(cublasCreate(&handle));

    // C = (1/Tb) * S * S^T
    // Col-major: C^T(Nn×Nn) = S^T(Nn×Tb) * S(Tb×Nn) — but S is symmetric so same
    float alpha=1.f/Tb, beta=0.f;

    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    // S is row-major Nn×Tb. cuBLAS sees it as col-major Tb×Nn.
    // We want C = S * S^T. In col-major: C = (S^T)^T * S^T
    // = S_colmajor_transposed * S_colmajor
    CUBLAS_CHECK(cublasSgemm(handle,
        CUBLAS_OP_T, CUBLAS_OP_N,
        Nn, Nn, Tb,
        &alpha, d_S, Tb, d_S, Tb, &beta, d_C, Nn));

    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    float gflops = 2.f*Nn*Nn*Tb/(ms*1e6f);
    printf("Covariance SGEMM: %.3f ms, %.1f GFLOPS\n",ms,gflops);

    // Save covariance matrix
    float* h_C=(float*)malloc((size_t)Nn*Nn*sizeof(float));
    CUDA_CHECK(cudaMemcpy(h_C,d_C,(size_t)Nn*Nn*sizeof(float),cudaMemcpyDeviceToHost));

    FILE* f=fopen("spike_cov.txt","w");
    fprintf(f,"# %d %d\n",Nn,Nn);
    for(int i=0;i<Nn;i++) { for(int j=0;j<Nn;j++) fprintf(f,"%.5f ",h_C[i*Nn+j]); fprintf(f,"\n"); }
    fclose(f);

    CUBLAS_CHECK(cublasDestroy(handle));
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_S); cudaFree(d_C);
    free(h_S); free(h_C);
    return 0;
}

In [ ]:
!nvcc -O2 -o spike_cov spike_cov.cu -lcublas && ./spike_cov 200 10000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load and visualize spike-train covariance matrix
with open('spike_cov.txt') as f:
    header = f.readline().split()
    N = int(header[1])
    C = np.array([list(map(float, line.split())) for line in f])

# Normalize to correlation
diag = np.sqrt(np.diag(C))
corr = C / np.outer(diag, diag)
np.fill_diagonal(corr, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(C, cmap='RdBu_r', aspect='auto')
plt.colorbar(im1, ax=axes[0])
axes[0].set_title('Spike-Train Covariance Matrix', fontsize=12)
axes[0].set_xlabel('Neuron j'); axes[0].set_ylabel('Neuron i')

im2 = axes[1].imshow(corr, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im2, ax=axes[1])
axes[1].set_title('Spike-Train Correlation Matrix', fontsize=12)
axes[1].set_xlabel('Neuron j'); axes[1].set_ylabel('Neuron i')

# Off-diagonal correlation distribution
off_diag = corr[np.triu_indices(N, k=1)]
print(f"Off-diagonal correlations: mean={off_diag.mean():.4f}, "
      f"std={off_diag.std():.4f}")
print(f"(Expected ~0 for independent Poisson neurons)")

plt.tight_layout()
plt.savefig('spike_covariance.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Task | cuBLAS function | Notes |
|------|----------------|-------|
| Rate network forward pass | `cublasSgemm` | W[M×N] × r[N×T] |
| Spike covariance | `cublasSgemm` | S[N×T] × S^T[T×N] |
| Single timestep update | `cublasSgemv` | y = W × x |
| Outer product (Hebbian) | `cublasSger` | A += α × x × y^T |

**Key insight:** For matrix sizes N ≥ 512, cuBLAS achieves 10–100× more GFLOPS than hand-written kernels because it uses tensor cores and optimal memory access patterns developed by NVIDIA engineers.

**Next lecture:** cuFFT for LFP spectral analysis.